<a href="https://colab.research.google.com/github/Ernest-Trevis/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Practical Exercise 4: Developing a Knowledge Based Agent**

Setup to run first:


In [1]:
!mkdir -p trailer data tests

In [2]:
%%writefile trailer/__init__.py
"""Self-driving trailer: a knowledge-based agent for freight routing in Kenya."""

Writing trailer/__init__.py


**Knowledge Base:**

In [3]:
%%writefile data/network.json
{
  "_note": "Road distances (km) are approximate. Replace with your Practical Exercise 3 graph, or refresh via trailer/api_loader.py.",
  "places": [
    {"name": "Mombasa",     "lat": -4.0435, "lon": 39.6682, "fuel": true,  "rest": true},
    {"name": "Voi",         "lat": -3.3963, "lon": 38.5560, "fuel": true,  "rest": true},
    {"name": "Mtito Andei", "lat": -2.6906, "lon": 38.1646, "fuel": true,  "rest": true},
    {"name": "Nairobi",     "lat": -1.2921, "lon": 36.8219, "fuel": true,  "rest": true},
    {"name": "Thika",       "lat": -1.0332, "lon": 37.0693, "fuel": true,  "rest": true},
    {"name": "Nyeri",       "lat": -0.4167, "lon": 36.9500, "fuel": true,  "rest": true},
    {"name": "Naivasha",    "lat": -0.7172, "lon": 36.4310, "fuel": false, "rest": false},
    {"name": "Nakuru",      "lat": -0.3031, "lon": 36.0800, "fuel": true,  "rest": true},
    {"name": "Kericho",     "lat": -0.3689, "lon": 35.2863, "fuel": false, "rest": true},
    {"name": "Eldoret",     "lat":  0.5143, "lon": 35.2698, "fuel": true,  "rest": true},
    {"name": "Kisumu",      "lat": -0.0917, "lon": 34.7680, "fuel": true,  "rest": true},
    {"name": "Malaba",      "lat":  0.6360, "lon": 34.2740, "fuel": true,  "rest": true}
  ],
  "roads": [
    {"from": "Mombasa",     "to": "Voi",         "km": 160, "speed_kmh": 80, "road": "A109"},
    {"from": "Voi",         "to": "Mtito Andei", "km": 110, "speed_kmh": 80, "road": "A109"},
    {"from": "Mtito Andei", "to": "Nairobi",     "km": 235, "speed_kmh": 80, "road": "A109"},
    {"from": "Nairobi",     "to": "Thika",       "km": 45,  "speed_kmh": 80, "road": "A2"},
    {"from": "Thika",       "to": "Nyeri",       "km": 100, "speed_kmh": 80, "road": "A2"},
    {"from": "Nyeri",       "to": "Nakuru",      "km": 160, "speed_kmh": 70, "road": "B5", "max_tonnes": 30},
    {"from": "Nairobi",     "to": "Naivasha",    "km": 90,  "speed_kmh": 80, "road": "A104"},
    {"from": "Naivasha",    "to": "Nakuru",      "km": 65,  "speed_kmh": 80, "road": "A104"},
    {"from": "Nakuru",      "to": "Kericho",     "km": 100, "speed_kmh": 70, "road": "A104"},
    {"from": "Kericho",     "to": "Kisumu",      "km": 75,  "speed_kmh": 70, "road": "A1", "max_tonnes": 25},
    {"from": "Nakuru",      "to": "Eldoret",     "km": 155, "speed_kmh": 80, "road": "A104"},
    {"from": "Eldoret",     "to": "Kisumu",      "km": 115, "speed_kmh": 70, "road": "B1"},
    {"from": "Eldoret",     "to": "Malaba",      "km": 120, "speed_kmh": 80, "road": "A104"}
  ]
}

Writing data/network.json


In [4]:
%%writefile trailer/kb.py

from __future__ import annotations

import json
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Optional

DEFAULT_DATA = Path(__file__).resolve().parent.parent / "data" / "network.json"


@dataclass(frozen=True)
class Place:
    name: str
    lat: float
    lon: float
    fuel: bool = False
    rest: bool = False


@dataclass(frozen=True)
class Edge:
    src: str
    dst: str
    km: float
    speed_kmh: int = 80
    road: str = ""
    max_tonnes: Optional[float] = None

    @property
    def hours(self) -> float:
        return self.km / self.speed_kmh


def _norm(x) -> str:
    return str(x).strip().lower().replace("_", " ")


class KnowledgeBase:
    def __init__(self) -> None:
        self.places: dict[str, Place] = {}
        self._out: dict[str, dict[str, Edge]] = {}
        self._dynamic: set[tuple] = set()

    def add_place(self, place: Place) -> None:
        self.places[place.name] = place
        self._out.setdefault(place.name, {})

    def add_edge(self, edge: Edge, oneway: bool = False) -> None:
        for n in (edge.src, edge.dst):
            if n not in self.places:
                raise KeyError(f"Unknown place: {n}")
        self._out[edge.src][edge.dst] = edge
        if not oneway:
            self._out[edge.dst][edge.src] = replace(edge, src=edge.dst, dst=edge.src)

    @classmethod
    def from_json(cls, path=None) -> "KnowledgeBase":
        data = json.loads(Path(path or DEFAULT_DATA).read_text(encoding="utf-8"))
        kb = cls()
        for p in data["places"]:
            kb.add_place(Place(p["name"], p["lat"], p["lon"], p.get("fuel", False), p.get("rest", False)))
        for r in data["roads"]:
            kb.add_edge(
                Edge(r["from"], r["to"], r["km"], r.get("speed_kmh", 80), r.get("road", ""), r.get("max_tonnes")),
                oneway=r.get("oneway", False),
            )
        return kb

    def resolve(self, name: str) -> str:
        for p in self.places:
            if _norm(p) == _norm(name):
                return p
        raise KeyError(f"Unknown place: {name!r}. Known: {', '.join(sorted(self.places))}")

    def neighbors(self, name: str) -> list[Edge]:
        return list(self._out.get(name, {}).values())

    def edge(self, a: str, b: str) -> Optional[Edge]:
        return self._out.get(a, {}).get(b)

    def edges(self) -> list[Edge]:
        return [e for out in self._out.values() for e in out.values()]

    def as_facts(self) -> set[tuple]:
        facts: set[tuple] = set()
        for p in self.places.values():
            facts.add(("place", p.name))
            facts.add(("at", p.name, p.lat, p.lon))
            if p.fuel:
                facts.add(("fuel_station", p.name))
            if p.rest:
                facts.add(("rest_stop", p.name))
        for e in self.edges():
            facts.add(("road", e.src, e.dst, e.km))
            facts.add(("speed_limit", e.src, e.dst, e.speed_kmh))
            if e.max_tonnes is not None:
                facts.add(("max_tonnes", e.src, e.dst, e.max_tonnes))
        return facts | self._dynamic

    def tell(self, *fact) -> None:
        self._dynamic.add(tuple(fact))

    def retract(self, *fact) -> None:
        self._dynamic.discard(tuple(fact))

    def ask(self, *pattern) -> list[tuple]:
        out = []
        for f in sorted(self.as_facts(), key=str):
            if len(f) != len(pattern):
                continue
            if all(p is None or p == "*" or _norm(p) == _norm(v) for p, v in zip(pattern, f)):
                out.append(f)
        return out

    def close(self, a: str, b: str, both: bool = True) -> None:
        self.tell("closed", a, b)
        if both:
            self.tell("closed", b, a)

    def reopen(self, a: str, b: str, both: bool = True) -> None:
        self.retract("closed", a, b)
        if both:
            self.retract("closed", b, a)

Writing trailer/kb.py


In [5]:
%%writefile trailer/distances.py

from __future__ import annotations

import math

EARTH_R = 6371.0088


def _xy(a, b):
    mean_lat = math.radians((a.lat + b.lat) / 2)
    dx = EARTH_R * math.radians(b.lon - a.lon) * math.cos(mean_lat)
    dy = EARTH_R * math.radians(b.lat - a.lat)
    return dx, dy


def haversine_km(a, b) -> float:
    p1, p2 = math.radians(a.lat), math.radians(b.lat)
    dphi, dl = p2 - p1, math.radians(b.lon - a.lon)
    h = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * EARTH_R * math.asin(math.sqrt(h))


def euclidean_km(a, b) -> float:
    return math.hypot(*_xy(a, b))


def manhattan_km(a, b) -> float:
    dx, dy = _xy(a, b)
    return abs(dx) + abs(dy)


def road_km(kb, path) -> float:
    return sum(kb.edge(a, b).km for a, b in zip(path, path[1:]))


def compare(kb, a: str, b: str, path=None) -> dict:
    pa, pb = kb.places[a], kb.places[b]
    out = {
        "haversine_km": round(haversine_km(pa, pb), 1),
        "euclidean_km": round(euclidean_km(pa, pb), 1),
        "manhattan_km": round(manhattan_km(pa, pb), 1),
    }
    if path:
        out["road_km"] = round(road_km(kb, path), 1)
    return out

Writing trailer/distances.py


In [6]:
%%writefile trailer/api_loader.py

from __future__ import annotations

import json
import time
import urllib.request
from dataclasses import replace

OSRM = "https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=false"


def fetch_road(a, b, timeout: int = 10):
    url = OSRM.format(lon1=a.lon, lat1=a.lat, lon2=b.lon, lat2=b.lat)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "strathmore-trailer-kba/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            route = json.load(r)["routes"][0]
        return route["distance"] / 1000, route["duration"] / 3600
    except Exception as exc:  # network down, rate limited, bad JSON
        print(f"[api_loader] {a.name}->{b.name} failed: {exc}")
        return None


def refresh_distances(kb, pause: float = 1.0) -> int:
    updated = 0
    for e in kb.edges():
        got = fetch_road(kb.places[e.src], kb.places[e.dst])
        if got:
            kb._out[e.src][e.dst] = replace(e, km=round(got[0], 1))
            updated += 1
        time.sleep(pause)
    return updated


if __name__ == "__main__":
    from .kb import KnowledgeBase

    kb = KnowledgeBase.from_json()
    print("updated", refresh_distances(kb), "of", len(kb.edges()), "directed edges")


Writing trailer/api_loader.py


In [7]:
from trailer.kb import KnowledgeBase
from trailer.distances import compare
kb = KnowledgeBase.from_json()
print(len(kb.places), 'places,', len(kb.edges()), 'directed edges')
print(kb.ask('fuel_station', '*'))
print(compare(kb, 'Nairobi', 'Mombasa'))

12 places, 26 directed edges
[('fuel_station', 'Eldoret'), ('fuel_station', 'Kisumu'), ('fuel_station', 'Malaba'), ('fuel_station', 'Mombasa'), ('fuel_station', 'Mtito Andei'), ('fuel_station', 'Nairobi'), ('fuel_station', 'Nakuru'), ('fuel_station', 'Nyeri'), ('fuel_station', 'Thika'), ('fuel_station', 'Voi')]
{'haversine_km': 439.9, 'euclidean_km': 439.9, 'manhattan_km': 622.1}
